In [ ]:
import os
import sys
import subprocess
from rdkit import Chem
from rdkit.Chem import rdFMCS
from rdkit.Chem import rdchem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
import numpy as np
from copy import deepcopy
from openbabel import pybel

# This script processes all .mol2 files in a specified directory using the cgenff tool

mol2_dir = "/home/raheelx/cphmd_walkthrough/mol2_Epik"
output_dir = "/home/raheelx/cphmd_walkthrough/cgenff_output"
os.makedirs(output_dir, exist_ok=True)

for filename in sorted(os.listdir(mol2_dir)):
    if filename.endswith(".mol2"):
        mol2_path = os.path.join(mol2_dir, filename)
        basename = filename.replace(".mol2", "")
        output_str = os.path.join(output_dir, f"{basename}.str")
        cmd = f"module load cgenff && cgenff -a < {mol2_path} > {output_str}"
        try:
            subprocess.run(cmd, shell=True, executable="/bin/bash", check=True)
            print(f"Processed {filename}")
        except subprocess.CalledProcessError as e:
            print(f"Failed processing {filename}: {e}")



CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...


Processed riboflavin_1.mol2
Processed riboflavin_2.mol2


CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
attype warning: enolate not explicitly supported
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...


Processed riboflavin_3.mol2
Processed riboflavin_4.mol2


CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
attype warning: enolate not explicitly supported
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...


Processed riboflavin_5.mol2
Processed riboflavin_6.mol2


In [ ]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem, rdFMCS


# Paths - adjust these as needed
sdf_dir = "/home/raheelx/cphmd_walkthrough/sdf_converted"   # Original converted sdf files dir
aligned_dir = "/home/raheelx/cphmd_walkthrough/sdf_aligned"  # Directory to save aligned sdf files
mol2_aligned_dir = "/home/raheelx/cphmd_walkthrough/mol2_aligned"  # Directory to save final mol2 aligned files

# Make sure output dirs exist
os.makedirs(aligned_dir, exist_ok=True)
os.makedirs(mol2_aligned_dir, exist_ok=True)

# Load all sdf files
sdf_files = sorted([f for f in os.listdir(sdf_dir) if f.endswith(".sdf")])
mols = []
for filename in sdf_files:
    sdf_path = os.path.join(sdf_dir, filename)
    mol = Chem.MolFromMolFile(sdf_path, removeHs=False)
    if mol is None:
        print(f"WARNING: Failed to load {filename} with RDKit, skipping.")
    else:
        mols.append((filename, mol))

if len(mols) == 0:
    raise RuntimeError("No molecules loaded successfully. Check input files.")

# Use first molecule as reference for MCS and alignment
ref_name, ref_mol = mols[0]

print("Finding MCS for alignment...")

# Extract just the RDKit Mol objects for MCS
rdkit_mols = [mol for _, mol in mols]

# Find the MCS (maximum common substructure) across all molecules
mcs_result = rdFMCS.FindMCS(rdkit_mols, threshold=1.0, completeRingsOnly=True, ringMatchesRingOnly=True)

print(f"MCS SMARTS: {mcs_result.smartsString}")

mcs_mol = Chem.MolFromSmarts(mcs_result.smartsString)
if mcs_mol is None:
    raise RuntimeError("Failed to create molecule from MCS SMARTS.")

# Get MCS atom indices on the reference molecule
ref_match = ref_mol.GetSubstructMatch(mcs_mol)
if not ref_match:
    raise RuntimeError("Reference molecule does not contain MCS core.")

print("Aligning molecules on MCS core atoms...")

for filename, mol in mols:
    # Generate conformer if none present
    if not mol.GetNumConformers():
        AllChem.EmbedMolecule(mol)
        AllChem.UFFOptimizeMolecule(mol)

    # Get MCS atom indices on this molecule
    mol_match = mol.GetSubstructMatch(mcs_mol)
    if not mol_match:
        print(f"WARNING: Molecule {filename} does not contain MCS core; skipping alignment.")
        continue

    # Create atomMap for alignment: list of (mol_atom_idx, ref_atom_idx)
    atom_map = list(zip(mol_match, ref_match))

    rmsd = AllChem.AlignMol(mol, ref_mol, atomMap=atom_map)
    print(f"Aligned {filename} (RMSD on core atoms = {rmsd:.3f})")

    aligned_path = os.path.join(aligned_dir, filename)
    Chem.MolToMolFile(mol, aligned_path)

print(f"Alignment complete. Aligned files saved to {aligned_dir}")


# === Convert aligned SDF files to MOL2 with unique atom names ===

def assign_unique_atom_names(mol):
    """
    Assign unique atom names per atom in the RDKit mol.
    For example: C1, C2, O1, H1, etc.
    Ensures no '+' in atom names.
    """
    atom_type_counts = {}
    for atom in mol.GetAtoms():
        elem = atom.GetSymbol()
        count = atom_type_counts.get(elem, 0) + 1
        atom_type_counts[elem] = count
        atom_name = f"{elem}{count}"
        atom.SetProp("_TriposAtomName", atom_name)

print("Converting aligned SDF files to MOL2 with unique atom names...")

for filename in sorted(os.listdir(aligned_dir)):
    if filename.endswith(".sdf"):
        sdf_path = os.path.join(aligned_dir, filename)
        mol = Chem.MolFromMolFile(sdf_path, removeHs=False)
        if mol is None:
            print(f"Failed to read {filename}, skipping.")
            continue

        assign_unique_atom_names(mol)

        # Write temporary mol file for pybel to read
        tmp_mol_path = "/tmp/tmp_mol.mol"
        Chem.MolToMolFile(mol, tmp_mol_path)

        pybel_mol = next(pybel.readfile("mol", tmp_mol_path))

        # Overwrite atom names in pybel to use _TriposAtomName
        for i, atom in enumerate(pybel_mol.atoms):
            rdkit_atom = mol.GetAtomWithIdx(i)
            atom.name = rdkit_atom.GetProp("_TriposAtomName")

        mol2_out_path = os.path.join(mol2_aligned_dir, filename.replace(".sdf", ".mol2"))
        pybel_mol.write("mol2", mol2_out_path, overwrite=True)
        print(f"Converted {filename} -> {mol2_out_path} with unique atom names.")

print("MOL2 conversion complete. Files saved to:", mol2_aligned_dir)

Finding MCS for alignment...
MCS SMARTS: [#8&!R]-&!@[#6&!R](-&!@[#6&!R](-&!@[#7&R])(-&!@[#1&!R])-&!@[#1&!R])(-&!@[#6&!R](-&!@[#8&!R]-&!@[#1&!R])(-&!@[#6&!R](-&!@[#8&!R]-&!@[#1&!R])(-&!@[#6&!R](-&!@[#8&!R]-&!@[#1&!R])(-&!@[#1&!R])-&!@[#1&!R])-&!@[#1&!R])-&!@[#1&!R])-&!@[#1&!R]
Aligning molecules on MCS core atoms...
Aligned riboflavin_1.sdf (RMSD on core atoms = 0.000)
Aligned riboflavin_2.sdf (RMSD on core atoms = 0.000)
Aligned riboflavin_3.sdf (RMSD on core atoms = 0.000)
Aligned riboflavin_4.sdf (RMSD on core atoms = 0.000)
Aligned riboflavin_5.sdf (RMSD on core atoms = 0.000)
Aligned riboflavin_6.sdf (RMSD on core atoms = 0.000)
Alignment complete. Aligned files saved to /home/raheelx/cphmd_walkthrough/sdf_aligned
Converting aligned SDF files to MOL2 with unique atom names...
Converted riboflavin_1.sdf -> /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_1.mol2 with unique atom names.
Converted riboflavin_2.sdf -> /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_2.mol2

In [24]:
# MCS code - imported from msld-py-prep workshop
# Ensure you have the msld_mcs.py file in the same directory or adjust the import
# path accordingly. This file should contain the MsldMCS class definition.
import sys
print(sys.executable)




/home/raheelx/miniforge3/envs/charmm/bin/python


In [ ]:
from msld_mcs import MsldMCS

MsldMCS(
    molfile="/home/raheelx/cphmd_walkthrough/mol2_aligned/mol_list.txt",
    mcsout="/home/raheelx/cphmd_walkthrough/mcs_results.txt",
    cutoff=0.8,
    debug=True
)
